In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[2]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

/Users/rawls/quant-lab


In [2]:
import pandas as pd
import numpy as np

from src.data.market_configs import MARKET_CONFIGS
from src.data.loader import download_market_data
from src.pipelines.strategy_returns import build_strategy_return_stack

from src.utils.metrics import (
    sharpe_ratio,
    max_drawdown,
    annualized_return,
)

from src.analysis.turnover import (
    compute_turnover,
    rolling_turnover,
    summarize_turnover,
    build_turnover_report,
)

In [3]:
#Deployment candidates

market_specs = {
    "India": {
        "config": MARKET_CONFIGS["india"],
        "signals": [
            "mr_ret_10",
            "low_vol_20",
            "mr_lowvol_blend",
        ],
    },
    "Brazil": {
        "config": MARKET_CONFIGS["brazil"],
        "signals": [
            "mom_blend",
            "mr_lowvol_blend",
        ],
    },
    "Japan": {
        "config": MARKET_CONFIGS["japan"],
        "signals": [
            "mr_ret_5",
            "mr_ret_20",
        ],
    },
}

In [10]:
from src.analysis.robustness import run_market_robustness

rebalance_tc_results = []

for market_name, spec in market_specs.items():
    spec = dict(spec)
    spec["name"] = market_name

    result = run_market_robustness(
        market_spec=spec,
        target_vol=0.10,
        rebalance_frequencies=[1, 2, 5, 10, "weekly"],
        transaction_costs=[0, 2, 5, 10, 20, 50],
    )

    rebalance_tc_results.append(result)

rebalance_tc_results_df = (
    pd.concat(rebalance_tc_results, ignore_index=True)
    .round(3)
    .sort_values(["Market", "Rebalance Frequency", "Cost bps"])
)

rebalance_tc_results_df

,Market,Signal Combo,Rebalance Frequency,Cost bps,Sharpe,MDD,CAGR,Calmar,Mean Turnover,Median Turnover,P95 Turnover,% Trading Days
30,Brazil,mom_blend + mr_lowvol_blend,1,0,1.670,-0.259,0.162,0.624,0.512,0.5,0.80,99.631
31,Brazil,mom_blend + mr_lowvol_blend,1,2,1.390,-0.288,0.132,0.459,0.512,0.5,0.80,99.631
32,Brazil,mom_blend + mr_lowvol_blend,1,5,0.971,-0.347,0.089,0.257,0.512,0.5,0.80,99.631
33,Brazil,mom_blend + mr_lowvol_blend,1,10,0.273,-0.467,0.021,0.045,0.512,0.5,0.80,99.631
34,Brazil,mom_blend + mr_lowvol_blend,1,20,-1.120,-0.838,-0.102,-0.122,0.512,0.5,0.80,99.631
...,...,...,...,...,...,...,...,...,...,...,...,...
85,Japan,mr_ret_5 + mr_ret_20,weekly,2,0.746,-0.325,0.069,0.212,0.133,0.0,0.75,21.187
86,Japan,mr_ret_5 + mr_ret_20,weekly,5,0.641,-0.333,0.058,0.175,0.133,0.0,0.75,21.187
87,Japan,mr_ret_5 + mr_ret_20,weekly,10,0.465,-0.346,0.041,0.118,0.133,0.0,0.75,21.187
88,Japan,mr_ret_5 + mr_ret_20,weekly,20,0.115,-0.370,0.006,0.017,0.133,0.0,0.75,21.187


In [11]:
best_rebalance_by_cost = (
    rebalance_tc_results_df
    .sort_values(
        ["Market", "Cost bps", "Sharpe"],
        ascending=[True, True, False]
    )
    .groupby(["Market", "Cost bps"], as_index=False)
    .first()
)

best_rebalance_by_cost

,Market,Cost bps,Signal Combo,Rebalance Frequency,Sharpe,MDD,CAGR,Calmar,Mean Turnover,Median Turnover,P95 Turnover,% Trading Days
0,Brazil,0,mom_blend + mr_lowvol_blend,weekly,2.071,-0.245,0.221,0.901,0.147,0.00,0.833,20.993
1,Brazil,2,mom_blend + mr_lowvol_blend,weekly,1.996,-0.247,0.212,0.857,0.147,0.00,0.833,20.993
2,Brazil,5,mom_blend + mr_lowvol_blend,weekly,1.883,-0.251,0.198,0.792,0.147,0.00,0.833,20.993
3,Brazil,10,mom_blend + mr_lowvol_blend,weekly,1.694,-0.256,0.176,0.689,0.147,0.00,0.833,20.993
4,Brazil,20,mom_blend + mr_lowvol_blend,10,1.475,-0.306,0.154,0.503,0.084,0.00,0.857,10.029
5,Brazil,50,mom_blend + mr_lowvol_blend,10,0.829,-0.409,0.083,0.202,0.084,0.00,0.857,10.029
6,India,0,mr_ret_10 + low_vol_20 + mr_lowvol_blend,1,2.174,-0.220,0.243,1.105,0.286,0.25,0.500,97.454
7,India,2,mr_ret_10 + low_vol_20 + mr_lowvol_blend,1,2.033,-0.238,0.225,0.945,0.286,0.25,0.500,97.454
8,India,5,mr_ret_10 + low_vol_20 + mr_lowvol_blend,2,1.832,-0.238,0.200,0.838,0.191,0.00,0.556,49.815
9,India,10,mr_ret_10 + low_vol_20 + mr_lowvol_blend,2,1.596,-0.269,0.171,0.638,0.191,0.00,0.556,49.815


The combined rebalance frequency and transaction cost analysis demonstrates that the optimal implementation of the deployment strategies depends on both market characteristics and trading costs. For India, daily rebalancing produced the highest gross performance under negligible transaction costs; however, the optimal frequency shifted to a two-day rebalance once costs reached approximately 5 bps, reflecting the trade-off between alpha capture and turnover. Brazil exhibited significantly more persistent signals, with weekly rebalancing remaining optimal across low-cost environments before transitioning to a ten-day schedule under higher transaction costs. Japan displayed intermediate behaviour, favouring two-day rebalancing at low transaction costs and five-day rebalancing as costs increased. These findings show that rebalance frequency should be considered a deployable strategy parameter rather than a fixed design choice, with implementation decisions tailored to both market dynamics and expected execution costs.

In [6]:
from pathlib import Path

results_dir = Path("../results")
results_dir.mkdir(exist_ok=True)

rebalance_tc_results_df.to_csv(
    results_dir / "rebalance_transaction_cost_results.csv",
    index=False,
)

print("Saved rebalance_transaction_cost_results.csv")

Saved rebalance_transaction_cost_results.csv


In [7]:
best_rebalance_by_cost.to_csv(
    results_dir / "best_rebalance_by_transaction_cost.csv",
    index=False,
)

print("Saved best_rebalance_by_transaction_cost.csv")

Saved best_rebalance_by_transaction_cost.csv


In [8]:
deployment_rebalance_table = (
    best_rebalance_by_cost
    .pivot(
        index="Cost bps",
        columns="Market",
        values="Rebalance Frequency",
    )
)

deployment_rebalance_table

Market,Brazil,India,Japan
Cost bps,,,
0,weekly,1,2
2,weekly,1,2
5,weekly,2,2
10,10,2,5
20,10,2,5
50,10,5,5


In [9]:
deployment_rebalance_table.to_csv(
    results_dir / "deployment_rebalance_summary.csv"
)

print("Saved deployment_rebalance_summary.csv")

Saved deployment_rebalance_summary.csv
